# LSG50 Index – Pipeline Construction

This notebook builds and persists the LSG50 index using:

- S&P 500 universe
- Net income CAGR (5Y)
- Market capitalization percentile
- Composite CFO Index
- SQLite persistence layer

This notebook is executable top-to-bottom.

In [15]:
#Import and Config

# Core
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from datetime import date
import sys
sys.path.append("../src")

from fundamentals import fetch_fundamentals

# Config
DB_PATH = "sqlite:///lsg50.db"
TOP_N = 50

engine = create_engine(DB_PATH)
today = str(date.today())

print("Pipeline date:", today)

Pipeline date: 2026-03-03


## 1. Database Schema Initialization

Create required tables if they do not exist.

In [16]:
with engine.connect() as conn:

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS universe (
        symbol TEXT PRIMARY KEY,
        name TEXT,
        sector TEXT
    );
    """))

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS fundamentals_snapshot (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT,
        calculation_date TEXT,
        market_cap REAL,
        net_income_cagr_5y REAL,
        growth_pct REAL,
        size_pct REAL,
        cfo_index REAL,
        FOREIGN KEY(symbol) REFERENCES universe(symbol)
    );
    """))

    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS lsg50_composition (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT,
        calculation_date TEXT,
        weight REAL,
        cfo_index REAL,
        FOREIGN KEY(symbol) REFERENCES universe(symbol)
    );
    """))

print("Schema ready.")

Schema ready.


## 2. Build S&P 500 Universe

In [17]:
def build_universe():

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    df_sp500 = pd.read_html(response.text)[0]

    df_sp500["symbol"] = (
        df_sp500["Symbol"]
        .str.replace(".", "-", regex=False)
        .str.strip()
    )

    df_universe = df_sp500[["symbol", "Security", "GICS Sector"]].rename(
        columns={
            "Security": "name",
            "GICS Sector": "sector"
        }
    )

    return df_universe

df_universe = build_universe()
df_universe.to_sql("universe", engine, if_exists="replace", index=False)

print("Universe stored:", len(df_universe), "companies")

Universe stored: 503 companies


/var/folders/qz/lqkm42d57sxb3jcp56ssgksm0000gn/T/ipykernel_26906/840134309.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_sp500 = pd.read_html(response.text)[0]


## 3. Fundamental Metrics Calculation

In [18]:
def compute_percentiles(df):

    df["growth_pct"] = df["net_income_cagr_5y"].rank(pct=True)
    df["size_pct"] = df["market_cap"].rank(pct=True)

    df["cfo_index"] = 0.5 * df["growth_pct"] + 0.5 * df["size_pct"]

    return df

symbols = df_universe["symbol"].tolist()

results = []

for s in symbols:
    data = fetch_fundamentals(s)
    if data is not None:
        results.append(data)

df_master = pd.DataFrame(results)

df_master = compute_percentiles(df_master)

df_master["calculation_date"] = today

cols = [
    "symbol",
    "calculation_date",
    "market_cap",
    "net_income_cagr_5y",
    "growth_pct",
    "size_pct",
    "cfo_index"
]

df_master[cols].to_sql(
    "fundamentals_snapshot",
    engine,
    if_exists="append",
    index=False
)

print("Fundamental snapshot stored:", len(df_master))

Fundamental snapshot stored: 444


## 4. LSG50 Index Construction

This section builds the **LSG50 Index**, a proprietary index constructed from the top 50 companies ranked by the CFO Index score.

Methodology:
- Rank companies by `cfo_index`
- Select top 50 constituents
- Assign weights proportional to their CFO score
- Store index composition in the database

In [19]:
TOP_N = 50

df_index = (
    df_master
    .sort_values("cfo_index", ascending=False)
    .head(TOP_N)
    .copy()
)

print("Selected constituents:", len(df_index))
df_index[["symbol", "cfo_index"]].head()

Selected constituents: 50


,symbol,cfo_index
303,NVDA,0.998874
179,GE,0.969595
253,LLY,0.956081
273,META,0.944820
380,TMUS,0.936937


### 4.2 Weight Allocation

Instead of equal weighting or market-cap weighting (like the S&P 500),  
the LSG50 assigns weights proportionally to the CFO Index score.

This reinforces the proprietary ranking methodology and creates
a differentiated index construction approach.

In [20]:
df_index["weight"] = df_index["cfo_index"] / df_index["cfo_index"].sum()

print("Weight sum check:", df_index["weight"].sum())
df_index[["symbol", "cfo_index", "weight"]].head()

Weight sum check: 1.0


,symbol,cfo_index,weight
303,NVDA,0.998874,0.023281
179,GE,0.969595,0.022598
253,LLY,0.956081,0.022283
273,META,0.944820,0.022021
380,TMUS,0.936937,0.021837


### 4.3 Sector Exposure (Sanity Check)

We analyze sector concentration to understand structural biases
relative to the broader S&P universe.

In [21]:
df_index = df_index.merge(
    df_universe[["symbol", "sector"]],
    on="symbol",
    how="left"
)

df_index["sector"].value_counts(normalize=True)

sector
Information Technology    0.22
Financials                0.18
Communication Services    0.14
Industrials               0.12
Health Care               0.12
Consumer Staples          0.08
Real Estate               0.04
Consumer Discretionary    0.04
Materials                 0.02
Utilities                 0.02
Energy                    0.02
Name: proportion, dtype: float64

### 4.4 Persist Index Composition

The final index composition is stored in the database,
including calculation date and assigned weights.

In [22]:
df_index["calculation_date"] = today

df_index[["symbol", "calculation_date", "weight", "cfo_index"]].to_sql(
    "lsg50_composition",
    engine,
    if_exists="append",
    index=False
)

print("LSG50 composition stored:", len(df_index))

LSG50 composition stored: 50


---

## LSG50 Construction Complete

The index has been successfully constructed and persisted.

Key characteristics:
- Top 50 by proprietary CFO Index
- Score-weighted allocation
- Snapshot stored for historical tracking